In [1]:
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

In [2]:
file_path = "/content/sample_data/raster_run1.csv"

In [3]:
cols = ["test","row","state","t_us","x","y","theta","r0","r1","r2","r3","r4"]
df = pd.read_csv(f"{file_path}", names=cols, header=0)
df = df[df.state == 1].reset_index(drop=True)       # forward passes only

# --- Sensor geometry (MEASURE THESE) ---
# lateral offset of each sensor from the robot centre line, mm, left positive
s = np.array([-18.0, -9.0, 0.0, 9.0, 18.0])         # placeholder values
f = 20.0                                             # sensor row ahead of the axle, mm (placeholder)
# Check that index 0..4 matches DN1..DN5 and which side is "left"

# --- Per-channel normalisation from your Sensors exercises ---
white = np.array([600, 620, 590, 610, 605], float)   # placeholder: your white baselines (µs)
black = np.array([2400, 2500, 2350, 2450, 2420], float)   # placeholder: your black baselines (µs)

pts = []
for i in range(5):
    r = df[f"r{i}"].to_numpy(float)
    # darkness 0 (white) .. 1 (black); discharge time grows with darkness
    dark = np.clip((r - white[i]) / (black[i] - white[i]), 0, 1)

    # sensor position in the world frame
    sx = df.x + f*np.cos(df.theta) - s[i]*np.sin(df.theta)
    sy = df.y + f*np.sin(df.theta) + s[i]*np.cos(df.theta)
    pts.append(pd.DataFrame({"x": sx, "y": sy, "dark": dark, "ch": i}))
pts = pd.concat(pts)

# --- Bin into a grid ---
cell = 5.0   # mm per pixel: try different sizes and compare
xi = np.floor((pts.x - pts.x.min()) / cell).astype(int)
yi = np.floor((pts.y - pts.y.min()) / cell).astype(int)
nx, ny = xi.max()+1, yi.max()+1

total = np.zeros((ny, nx)); count = np.zeros((ny, nx))
np.add.at(total, (yi, xi), pts.dark.to_numpy())
np.add.at(count, (yi, xi), 1)

img = np.where(count > 0, total / np.maximum(count, 1), np.nan)
coverage = (count > 0).mean()
print(f"coverage: {coverage:.1%}")

plt.imshow(img, origin="lower", cmap="gray_r", extent=[0, nx*cell, 0, ny*cell])
plt.colorbar(label="darkness (0 white – 1 black)")
plt.show()

TypeError: 'float' object cannot be interpreted as an integer